In [ ]:
# 第9周-Day3：Deployment / DeploymentRevision — 为什么 Deployment 独立于 Release？
# matplotlib 中文字体配置
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("使用字体:", font_name)


## 📅 Week 9 - Day 3 | 2026-07-29

### Deployment / DeploymentRevision — 为什么 Deployment 独立于 Release？

| 项目 | 内容 |
|---|---|
| **本周主题** | Domain Deep Dive — 逐一拆解 LangChat 核心对象 |
| **今日主题** | Deployment 和 DeploymentRevision：为什么部署独立于发布？ |
| **学习时间** | 30 分钟 |
| **连接主线** | W9-D2（SkillRelease 制品）→ **D3（Deployment 运行时）** → D4（ReleaseChannel/TrafficPolicy） |


## ❓ 今日核心问题

### 为什么 Deployment 独立于 Release？

**为什么不是：SkillRelease 一旦发布（Published），就直接跑起来？**

这击中了 LangChat 架构里一个极其重要的分离：**制品（Artifact）和部署（Deployment）是两个生命周期。**

**一句话：Release 回答"这是什么"，Deployment 回答"在哪儿、怎么跑、跑哪个版本"。**


## 🗣 人话解释（Jason 26年 ERP 经验）

SAP Enhancement Package 发布了（= SkillRelease Published），但你不能直接让客户用它。你需要：

1. **确认兼容性**（→ Compatibility Matrix）
2. **绑定具体环境**（→ environment / scope）
3. **配置数据库连接**（→ binding_manifest_digest）
4. **决定全量还是灰度**（→ TrafficPolicy）
5. **生成上线快照**（→ DeploymentRevision）

这个过程就是 **materialization（物化）**：把通用制品变成特定环境的可执行实例。


In [ ]:
# LangChat 四层架构中 Deployment 的位置
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Layers
layers = [
    (8, '#E8F5E9', 'Business Domain Layer\nDigitalEmployeeDefinition → ApplicationContractVersion'),
    (6, '#E3F2FD', 'Supply Chain Layer\nBlueprint → Build → SkillRelease → ReleaseChannel'),
    (4, '#FFF3E0', 'Runtime Layer  ◄── 今天焦点\nDeployment → DeploymentRevision → TrafficPolicy → FrozenExecutionContext'),
    (2, '#F3E5F5', 'Operations Layer\nRegistry / Catalog Projection'),
]
for y, color, text in layers:
    rect = plt.Rectangle((0.5, y-0.8), 9, 1.6, facecolor=color, edgecolor='#333', linewidth=1.5, rounded=True)
    ax.add_patch(rect)
    ax.text(5, y, text, ha='center', va='center', fontsize=9, fontweight='bold')

# Arrow connecting Supply Chain to Runtime via DeploymentRevision
ax.annotate('', xy=(5, 4.8), xytext=(5, 5.2),
            arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.text(6.2, 5.0, 'DeploymentRevision\n(唯一桥梁)', fontsize=8, color='red', fontweight='bold',
        ha='center', va='center', style='italic')

ax.set_title('LangChat 四层架构 — DeploymentRevision 是 Supply Chain 与 Runtime 的唯一桥梁', fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/tmp/w9d3_arch.png', dpi=150, bbox_inches='tight')
plt.show()
print("架构图已生成")


## 📋 ADR 依据

### Domain Model §8.1 RT-01 / RT-02

**Deployment（RT-01）**：
- 部署聚合，承载 DigitalEmployeeDefinition 在某 scope 内的部署生命周期
- 生命周期：`Draft → Active → Suspended → Retired`

**DeploymentRevision（RT-02）**：
- 完整运行时闭包，digest-pin 一切影响执行结果的内容
- **唯一可被流量路由的对象**
- 16 字段闭包：skill_release_digest、application_contract_version、runtime_abi_version 等
- 不可变量：完整闭包、不可修改、唯一可路由性

### 跨对象不变量（§10.4）
- 不变量 2：闭包完整性
- 不变量 3：流量精确性
- 不变量 4：Channel 与流量解耦
- 不变量 8：回滚前向性


In [ ]:
# DeploymentRevision 16 字段闭包可视化
fig, ax = plt.subplots(figsize=(14, 9))
ax.axis('off')

fields = [
    ('skill_release_digest', 'SkillRelease 精确版本', '#E91E63'),
    ('application_contract_version', '应用契约版本', '#9C27B0'),
    ('runtime_abi_version', 'Runtime ABI 版本', '#673AB7'),
    ('runtime_profile', '运行时配置 Profile', '#3F51B5'),
    ('manifest_schema_version', 'Manifest Schema 版本', '#2196F3'),
    ('execution_plan_ir_schema_version', '执行计划 IR Schema', '#03A9F4'),
    ('frozen_context_schema_version', '冻结上下文 Schema', '#00BCD4'),
    ('required_artifact_media_types', '所需 Artifact 类型', '#009688'),
    ('knowledge_snapshot_digests', '知识库快照', '#4CAF50'),
    ('capability_release_digests', '能力发布 digest', '#8BC34A'),
    ('policy_bundle_digest', '策略包 digest', '#CDDC39'),
    ('prompt_artifacts', 'Prompt 制品', '#FFC107'),
    ('model_artifacts', '模型制品', '#FF9800'),
    ('runtime_artifacts', '运行时制品', '#FF5722'),
    ('environment', '环境绑定', '#795548'),
    ('binding_manifest_digest', '绑定清单 digest', '#607D8B'),
]

# Draw as a grid
for i, (field, desc, color) in enumerate(fields):
    row = i // 4
    col = i % 4
    x = col * 3.3 + 0.5
    y = 8 - row * 2.0
    
    rect = plt.Rectangle((x, y-0.7), 3.0, 1.5, facecolor=color, alpha=0.15, edgecolor=color, linewidth=2, rounded=True)
    ax.add_patch(rect)
    ax.text(x+1.5, y+0.25, field, ha='center', va='center', fontsize=6.5, fontweight='bold', color=color)
    ax.text(x+1.5, y-0.35, desc, ha='center', va='center', fontsize=7, color='#555')

ax.set_xlim(0, 14)
ax.set_ylim(-1, 10)
ax.set_title('DeploymentRevision 闭包：16 个字段锁死一切执行结果', fontsize=13, fontweight='bold', pad=15)

# Note at bottom
ax.text(7, -0.5, '⚠️ source_channel 和 evaluation_only 不参与 digest 计算（AS §11.3）',
        ha='center', fontsize=8, color='red', style='italic')

plt.tight_layout()
plt.savefig('/tmp/w9d3_closure.png', dpi=150, bbox_inches='tight')
plt.show()
print("16字段闭包图已生成")


## 🔍 代码验证（/root/langchat）

### DeploymentRevision dataclass
- ✅ `frozen=True` dataclass 保证 Python 层面不可变
- ✅ 16 字段闭包与 AS §11.2 逐项匹配
- ✅ `source_channel` 和 `evaluation_only` 不参与 digest

### Deployment aggregate
- ✅ 是生命周期聚合，只持有 Revision 引用（不是内容）
- ✅ `add_reference` 拒绝 evaluation-only Revision（安全门）

### 持久化模型（Append-Only）
- ✅ 新 Revision = 新行，不修改旧行
- ✅ 回滚通过标记 `is_active` 实现（前向语义）


In [ ]:
# LangChat Deployment → MI CRE 场景映射
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

data = [
    ['SkillRelease', 'MI 标准合同查询数字员工 v2.3（已签名包）', '通用能力包'],
    ['Deployment', '客户A购物中心使用的"合同查询数字员工"', '特定客户部署实例'],
    ['DeploymentRevision', '今天在客户A production 上线的具体快照', '绑定 ERP API、租户权限、知识库'],
    ['binding_manifest', '客户A的 SAP IP、端口、API Key 版本', '环境绑定（不含明文 secret）'],
    ['TrafficPolicy', '80%→旧版，20%→新版', '灰度策略'],
    ['回滚（前向）', '物化历史闭包为新 Revision', '不还原，创造新决策记录'],
    ['evaluation_only', '隔离环境测试新版本', '评估 Revision 不可被生产引用'],
]

colors = ['#E8F5E9', '#E3F2FD', '#FFF3E0', '#F3E5F5', '#FFFDE7', '#FBE9E7', '#E0F2F1']
for i, (lang, cre, desc) in enumerate(data):
    y = 6 - i * 0.85
    ax.fill_between([0.5, 13.5], y-0.3, y+0.3, color=colors[i], alpha=0.7)
    ax.text(2.0, y, lang, fontsize=9, fontweight='bold', ha='center', va='center')
    ax.text(6.5, y, cre, fontsize=8, ha='center', va='center')
    ax.text(11.0, y, desc, fontsize=8, ha='center', va='center', color='#555')

# Header
ax.fill_between([0.5, 13.5], 6.7, 7.1, color='#37474F', alpha=0.9)
ax.text(2.0, 6.9, 'LangChat', fontsize=10, fontweight='bold', ha='center', va='center', color='white')
ax.text(6.5, 6.9, 'MI CRE 场景', fontsize=10, fontweight='bold', ha='center', va='center', color='white')
ax.text(11.0, 6.9, '说明', fontsize=10, fontweight='bold', ha='center', va='center', color='white')

ax.set_xlim(0, 14)
ax.set_ylim(-0.5, 7.5)
ax.set_title('LangChat Deployment/DeploymentRevision → MI CRE 场景映射', fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/tmp/w9d3_cre_mapping.png', dpi=150, bbox_inches='tight')
plt.show()
print("CRE映射图已生成")


In [ ]:
# 三种方案对比：为什么 Deployment 独立于 Release？
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')

dimensions = ['部署表示', '环境绑定', '依赖锁定', '兼容矩阵', '回滚', '灰度', '审计', '可移植性']
plan_a = ['Release tag', '写死在 Release', '不锁定(latest)', '不检查', '切回旧tag', '困难', '部署日志', '差']
plan_b = ['Release+配置', '运行时注入', '部分锁定', 'CI脚本', '重新部署', '需额外工具', 'Pipeline记录', '中']
plan_c = ['DeploymentRevision闭包', 'digest-pin', '完全锁定', '闭包内含', '物化新Revision', '原生支持', '不可变对象', '高']

# Highlight plan C as the winner
for i, dim in enumerate(dimensions):
    y = 7 - i * 0.9
    ax.text(1.5, y, dim, fontsize=9, fontweight='bold', ha='center', va='center')
    ax.text(5.0, y, plan_a[i], fontsize=8, ha='center', va='center', color='#E53935')
    ax.text(8.5, y, plan_b[i], fontsize=8, ha='center', va='center', color='#FB8C00')
    ax.text(12.0, y, plan_c[i], fontsize=8, ha='center', va='center', color='#43A047', fontweight='bold')
    
    # Background
    bg_color = '#FFF8E1' if i % 2 == 0 else '#FFFFFF'
    ax.fill_between([0.3, 13.5], y-0.35, y+0.35, color=bg_color, alpha=0.5)

# Headers
ax.fill_between([0.3, 13.5], 7.5, 7.9, color='#263238', alpha=0.9)
ax.text(1.5, 7.7, '维度', fontsize=10, fontweight='bold', ha='center', va='center', color='white')
ax.text(5.0, 7.7, 'A: Release直接跑', fontsize=10, fontweight='bold', ha='center', va='center', color='white')
ax.text(8.5, 7.7, 'B: Release+配置', fontsize=10, fontweight='bold', ha='center', va='center', color='white')
ax.text(12.0, 7.7, 'C: LangChat闭包 ✅', fontsize=10, fontweight='bold', ha='center', va='center', color='white')

ax.set_xlim(0, 14)
ax.set_ylim(-0.5, 8.3)
ax.set_title('核心原因：AI 应用执行结果不确定性远超传统软件，必须完全锁定', fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/tmp/w9d3_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("方案对比图已生成")


## 🧠 架构师思考题

**场景**：客户 A（20 个购物中心）运行"合同查询数字员工"。需求：
- 总部运行 v2.3
- 3 个新店试运行 v2.4-beta
- 其他 17 个店继续 v2.3
- v2.4-beta 不好 → 一键回滚

**请思考**：
1. 几个 SkillRelease？几个 Deployment？几个 DeploymentRevision？
2. "一键回滚"的具体操作序列是什么？
3. 新 Capability 的 digest-pin 时机？
4. "试运行"是 evaluation_only 还是生产灰度？

> 提示：关注 environment 字段和 TrafficPolicy 的 cohort 路由。


## 💡 我的理解变化

**以前以为**：Deployment 是运维动作，不是架构对象。Release 和 Deployment 是"发布"的前后两步。

**现在知道**：
1. DeploymentRevision 是 Runtime 层最重要的架构对象，不是运维概念
2. Release 和 Deployment 的分离是架构刚性约束，不是流程偏好
3. 回滚不是"还原"，而是"前进到一个与历史内容相同的新版本"
4. evaluation_only 是安全护栏，不是调试工具


## 📖 术语表

| 英文术语 | 音标 | 释义 |
|---|---|---|
| Deployment | /dɪˈplɔɪmənt/ | 部署聚合，承载生命周期 |
| DeploymentRevision | /dɪˈplɔɪmənt rɪˈvɪʒən/ | 完整运行时闭包，唯一可路由 |
| Materialization | /məˌtɪəriəlaɪˈzeɪʃən/ | 物化：通用制品→特定实例 |
| binding_manifest | /ˈbaɪndɪŋ ˈmænɪfest/ | 绑定清单（环境绑定） |
| TrafficPolicy | /ˈtræfɪk ˈpɒlɪsɪ/ | 流量路由策略 |
| evaluation_only | /ɪˌvæljuˈeɪʃən ˈoʊnli/ | 评估模式，不接生产流量 |
| digest-pin | /ˈdaɪdʒɛst pɪn/ | 精确锁定到内容摘要 |
| Append-Only | /əˈpɛnd ˈoʊnli/ | 只追加，不修改历史 |


## ✏️ 课堂练习

**Q1**: 同一个 SkillRelease digest，在客户 A 和客户 B 部署，生成的 DeploymentRevision digest 相同吗？

> 不同。环境绑定（environment）和 binding_manifest_digest 不同，闭包 digest 必然不同。

**Q2**: 回滚时，直接修改旧 DeploymentRevision 的 is_active 标记对吗？

> 不对。回滚是前向操作——物化新 Revision（内容与历史相同），切换 is_active。不修改旧行。

**Q3**: 为什么 source_channel 不参与 digest 计算？

> 因为 Channel 是 Supply Chain 的概念（可移动指针），DeploymentRevision 是 Runtime 对象。把 Channel 放进 digest 会导致 Channel 移动后历史 Revision 的 digest 变化，破坏不可变性。


## 🔗 明日连接

**Day4：ReleaseChannel / TrafficPolicy — 为什么需要灰度？**

今天学了 DeploymentRevision 是"完整运行时闭包"，明天学 TrafficPolicy 如何控制多个 Revision 之间的流量分配。

**Semantic Layer 定位**：
```
Ontology
  └─ Domain Model
       ├─ Deployment ──── 今天：部署生命周期聚合
       │    └─ DeploymentRevision ── 完整运行时闭包（16字段）
       ├─ ReleaseChannel ── 昨天：晋升指针（Supply Chain）
       ├─ TrafficPolicy ── 明天：流量路由策略（Runtime）
       └─ FrozenExecutionContext ── 后天：不可变执行上下文
```

DeploymentRevision 是两个世界（Supply Chain ↔ Runtime）之间唯一的合法通道。
